# Week 2 — SigLIP Joint Embeddings (SNLI-VE or MSCOCO)
**Project**: Quantum Kernels — Vision-Language Encoder Comparison

**Purpose:** the CLIP-vs-SigLIP comparison. Same joint-embedding multimodal tasks
you ran with CLIP, now with SigLIP (Google's sigmoid-loss vision-language model).
This tests whether the multimodal finding is CLIP-specific.

**SigLIP differences from CLIP:**
- Output dim is 768 per modality (vs CLIP's 512) -> joint = 1536-dim
- Text needs padding='max_length' (SigLIP was trained this way)
- Same get_image_features / get_text_features interface

**Set DATASET at the top.** Run once for snlive, once for mscoco.
**Metric:** macro F1 (SNLI-VE, 3-class) or binary F1 (MSCOCO, 2-class)
**Runtime:** ~15 min

In [ ]:
!pip install once in terminal: --quiet
import numpy as np, time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Config + local storage

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")
def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")
def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"; fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname); return dp if os.path.exists(dp) else fname
print("Helpers ready (Drive + download).")

---
## Step 1: Load dataset + collect images/texts/labels

Reuses the same sources you validated for CLIP:
- SNLI-VE: `sedrickkeh/snli-ve` (images bundled)
- MSCOCO:  `lmms-lab/COCO-Caption` (val split, images bundled, captions in 'answer')

In [ ]:
from datasets import load_dataset

if DATASET == 'snlive':
    print("Loading sedrickkeh/snli-ve...")
    ds = load_dataset("sedrickkeh/snli-ve")
    cols = ds['train'].column_names
    print(f"Columns: {cols}")
    img_col = 'image' if 'image' in cols else [c for c in cols if 'image' in c.lower() or 'img' in c.lower()][0]
    txt_col = 'hypothesis' if 'hypothesis' in cols else [c for c in cols if 'hypo' in c.lower() or 'sentence' in c.lower()][0]
    lbl_col = 'label' if 'label' in cols else [c for c in cols if 'label' in c.lower() or 'gold' in c.lower()][0]
    # SNLI-VE labels may be strings — map if needed
    STR2INT = {'entailment':0,'neutral':1,'contradiction':2}
    def get_label(s):
        l = s[lbl_col]
        return STR2INT.get(l, l) if isinstance(l, str) else l

    def load_balanced(split, n_per_class):
        imgs, txts, lbls = [], [], []; counts={0:0,1:0,2:0}
        for s in ds[split]:
            l = get_label(s)
            if l not in [0,1,2] or counts[l] >= n_per_class: continue
            img = s[img_col]
            if not isinstance(img, Image.Image): continue
            imgs.append(img.convert('RGB')); txts.append(s[txt_col]); lbls.append(l); counts[l]+=1
            if all(v>=n_per_class for v in counts.values()): break
        return imgs, txts, np.array(lbls)

    train_imgs, train_txts, train_labels = load_balanced('train', 400)
    test_split = 'test' if 'test' in ds else 'validation'
    test_imgs, test_txts, test_labels = load_balanced(test_split, 90)

elif DATASET == 'mscoco':
    print("Loading lmms-lab/COCO-Caption (streaming)...")
    ds_stream = load_dataset("lmms-lab/COCO-Caption", split="val", streaming=True)
    all_imgs, all_caps = [], []
    for s in ds_stream:
        img = s.get('image'); cap = s.get('answer')
        if isinstance(cap, list) and len(cap): cap = cap[0]
        if not isinstance(img, Image.Image): continue
        all_imgs.append(img.convert('RGB')); all_caps.append(str(cap))
        if len(all_imgs) >= 800: break
    # build matched/mismatched pairs
    rng = np.random.RandomState(42)
    def build_pairs(idxs):
        pairs=[]
        for i in idxs:
            pairs.append((i,i,1))
            j = rng.choice([k for k in idxs if k!=i]); pairs.append((i,int(j),0))
        rng.shuffle(pairs); return pairs
    tr_idx = list(range(600)); te_idx = list(range(600, 730))
    train_pairs = build_pairs(tr_idx); test_pairs = build_pairs(te_idx)
    # store for assembly after encoding
    print(f"Built {len(train_pairs)} train pairs, {len(test_pairs)} test pairs")

print("Dataset loaded.")

---
## Step 2: Load SigLIP + extract joint embeddings — checkpoint after

SigLIP outputs 768-dim per modality. Joint = 1536-dim.
Text uses padding='max_length' (required for SigLIP).

In [ ]:
MODEL_NAME = "google/siglip-base-patch16-224"
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print(f"SigLIP loaded. Per-modality dim: 768, joint: 1536")

def encode_images(images, batch_size=32):
    embs=[]
    for i in range(0, len(images), batch_size):
        inp = processor(images=images[i:i+batch_size], return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            e = model.get_image_features(**inp)
            e = e / torch.linalg.norm(e, dim=-1, keepdim=True)
        embs.append(e.cpu().numpy())
    return np.vstack(embs)

def encode_texts(texts, batch_size=32):
    embs=[]
    for i in range(0, len(texts), batch_size):
        inp = processor(text=texts[i:i+batch_size], return_tensors='pt',
                        padding='max_length', truncation=True, max_length=64).to(DEVICE)
        with torch.no_grad():
            e = model.get_text_features(**inp)
            e = e / torch.linalg.norm(e, dim=-1, keepdim=True)
        embs.append(e.cpu().numpy())
    return np.vstack(embs)

t0=time.time()
if DATASET == 'snlive':
    print("Encoding train (image + text)...")
    tr_img = encode_images(train_imgs); tr_txt = encode_texts(train_txts)
    train_emb = np.concatenate([tr_img, tr_txt], axis=1)
    te_img = encode_images(test_imgs); te_txt = encode_texts(test_txts)
    test_emb = np.concatenate([te_img, te_txt], axis=1)

elif DATASET == 'mscoco':
    print("Encoding all images + captions (cached)...")
    img_cache = encode_images(all_imgs)
    txt_cache = encode_texts(all_caps)
    def assemble(pairs):
        X,y=[],[]
        for ii,ci,lb in pairs:
            X.append(np.concatenate([img_cache[ii], txt_cache[ci]])); y.append(lb)
        return np.array(X), np.array(y)
    train_emb, train_labels = assemble(train_pairs)
    test_emb,  test_labels  = assemble(test_pairs)

print(f"Done {time.time()-t0:.0f}s | train {train_emb.shape} test {test_emb.shape}")
assert train_emb.shape[1] == 1536, f"Expected 1536-dim joint, got {train_emb.shape[1]}"
print("\nCheckpoint 1 — raw joint embeddings:")
checkpoint(f'{TAG}_raw.npz', lambda p: np.savez(p, train_emb=train_emb, test_emb=test_emb,
                                                train_labels=train_labels, test_labels=test_labels))

---
## Step 3: PCA (1536 → 8) + normalize

In [ ]:
N_QUBITS=8
pca_objects={n: PCA(n_components=n, random_state=42).fit(train_emb) for n in [4,8,12,16]}
print("PCA variance explained (1536-dim joint -> reduced):")
for n in [4,8,12,16]: print(f"  {n:>2} -> {np.sum(pca_objects[n].explained_variance_ratio_)*100:.1f}%")
pca_8=pca_objects[8]
X_train_pca=pca_8.transform(train_emb); X_test_pca=pca_8.transform(test_emb)
scaler=MinMaxScaler(feature_range=(0,np.pi))
X_train_scaled=scaler.fit_transform(X_train_pca)
X_test_scaled=np.clip(scaler.transform(X_test_pca),0,np.pi)
print(f"Train range: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")
checkpoint(f'{TAG}_pca_scaler.pkl', lambda p: pickle.dump(
    {'pca':pca_8,'scaler':scaler,'n_qubits':8,'embedding_type':'siglip_joint_1536','n_classes':CFG['n_classes']}, open(p,'wb')))

---
## Step 4: Classical baselines + save

In [ ]:
idx_tr=np.random.RandomState(42).choice(len(X_train_scaled),1000,replace=False)
idx_te=np.random.RandomState(42).choice(len(X_test_scaled),200,replace=False)
X_tr,y_tr=X_train_scaled[idx_tr],train_labels[idx_tr]
X_te,y_te=X_test_scaled[idx_te],test_labels[idx_te]
def f1m(yt,yp): return float(f1_score(yt,yp,average=AVG))
print(f"Train {X_tr.shape} Test {X_te.shape} | metric={AVG} F1")
classifiers={'Linear':SVC(kernel='linear',C=1.0,random_state=42),
             'RBF':SVC(kernel='rbf',C=1.0,gamma='scale',random_state=42),
             'Polynomial':SVC(kernel='poly',C=1.0,degree=3,coef0=1,random_state=42)}
results={}
print(f"{'Kernel':<14}{'Acc':>10}{'F1':>10}")
for name,clf in classifiers.items():
    clf.fit(X_tr,y_tr); yp=clf.predict(X_te)
    results[name]={'accuracy':float(accuracy_score(y_te,yp)),'f1':f1m(y_te,yp)}
    print(f"{name:<14}{results[name]['accuracy']:>10.4f}{results[name]['f1']:>10.4f}")
checkpoint(f'{TAG}_baseline_results.pkl', lambda p: pickle.dump(results,open(p,'wb')))

def save_final(p):
    np.savez(p, X_train=X_train_scaled, X_test=X_test_scaled, y_train=train_labels, y_test=test_labels,
             X_tr_subset=X_tr, X_te_subset=X_te, y_tr_subset=y_tr, y_te_subset=y_te)
checkpoint(f'{TAG}_embeddings.npz', save_final)
loaded=np.load(_find(f'{TAG}_embeddings.npz'))
assert loaded['X_tr_subset'].shape==(1000,8); assert loaded['X_te_subset'].shape==(200,8)
print("VERIFIED (1000,8)/(200,8).")
checkpoint(f'{TAG}_meta.json', lambda p: json.dump(
    {'dataset':DATASET,'encoder':'siglip','embedding_type':'siglip_joint_1536',
     'n_classes':CFG['n_classes'],'metric':CFG['metric'],'baselines':results}, open(p,'w'), indent=2))
print(f"\nWeek 2 SigLIP {DATASET} done. Tag: {TAG}")
print("Week 3/4: reuse your existing snlive/mscoco Week 3/4 notebooks, point _find at the _siglip tag files.")

---
## Done — SigLIP Week 2

### Saved (tag = `{TAG}`):
- `{TAG}_raw.npz`, `{TAG}_pca_scaler.pkl`, `{TAG}_baseline_results.pkl`,
  `{TAG}_embeddings.npz`, `{TAG}_meta.json`

### For Week 3/4:
Your existing `week3_snlive.ipynb` / `week3_mscoco.ipynb` work unchanged —
just change the loaded filename from e.g. `snlive_embeddings.npz` to
`snlive_siglip_embeddings.npz` in Step 1. Same quantum kernel, same metric.

### The comparison this enables:
CLIP-joint vs SigLIP-joint on the same task. If the quantum-vs-classical pattern
holds across both encoders, the multimodal finding is not CLIP-specific.